# Google Colab edition

Run the setup cell first, then work from top to bottom. Keep the course ZIP in My Drive. Use a fresh Colab runtime when switching dependency profiles. **Select a GPU runtime before running this notebook.** 

Adapted from your existing exercise. Previous local outputs were cleared so new Colab runs provide their own evidence. Run image notebook 00 first, using the same QUICK_RUN setting.

Source: Original local package: 03_fastai/03_compare_image_models.ipynb

In [ ]:
# Colab setup. Run this cell before the exercise cells.
from pathlib import Path
import os, sys, zipfile

USE_GOOGLE_DRIVE = True  #@param {type:"boolean"}
QUICK_RUN = False  #@param {type:"boolean"}
ARCHIVE_PATH = "/content/drive/MyDrive/HaiLong_Colab_Exercises_2026-10-03.zip"  #@param {type:"string"}

try:
    from google.colab import drive, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if USE_GOOGLE_DRIVE:
        if not Path('/content/drive/MyDrive').is_dir():
            drive.mount('/content/drive', timeout_ms=300000)
    COURSE_ROOT = Path('/content/HaiLong_Colab_Exercises')
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        archive = Path(ARCHIVE_PATH)
        if not archive.is_file():
            print('Select HaiLong_Colab_Exercises_2026-10-03.zip from your computer.')
            uploaded = files.upload()
            choices = [Path(n) for n in uploaded if n.endswith('.zip')]
            if len(choices) != 1:
                raise ValueError('Upload exactly one course ZIP file.')
            archive = choices[0]
        with zipfile.ZipFile(archive) as z:
            base = Path('/content').resolve()
            for member in z.infolist():
                target = (base / member.filename).resolve()
                if not target.is_relative_to(base) or not member.filename.startswith('HaiLong_Colab_Exercises/'):
                    raise ValueError('This is not the expected course archive.')
            z.extractall(base)
    if not (COURSE_ROOT / 'colab_runtime.py').is_file():
        raise FileNotFoundError('The ZIP must contain HaiLong_Colab_Exercises/colab_runtime.py.')
else:
    # Also permits local checks without assuming the current notebook directory.
    COURSE_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                        if (p / 'colab_runtime.py').is_file()), None)
    if COURSE_ROOT is None:
        raise RuntimeError('Open this notebook in Colab or run it inside the extracted course folder.')

os.environ['COURSE_SMOKE'] = '1' if QUICK_RUN else '0'
os.environ['WANDB_MODE'] = 'offline'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
sys.path.insert(0, str(COURSE_ROOT))
from colab_runtime import setup_course, finish_run
# Filled individually for each notebook by the builder.

COLAB_CONTEXT = setup_course(COURSE_ROOT, '03_fastai', 'fastai', '03_compare_image_models.ipynb', USE_GOOGLE_DRIVE, install=IN_COLAB)

## Full exercise evidence

- Run all exercise cells top to bottom in hosted Colab and preserve their outputs, errors and execution counts.
- Keep QUICK_RUN=False; shortened checks or synthetic replacements for real datasets do not count as full original exercises.
- Preserve plots, models, metrics and run configurations in Drive; a final-cell marker alone is insufficient.
- Use the real photographic branch and retain its split, class counts, held-out results and model artefacts.

This cell describes evidence to collect; it does not certify completion. GPU batch caps preserve dataset coverage and epoch counts, but are documented configuration changes.


# Lesson 3: compare image model configurations
Source task: https://github.com/fastai/course22/blob/master/03-which-image-models-are-best.ipynb
This is an independently written solution companion, not a copy of the instructor notebook or a course completion record. A controlled ResNet18/ResNet34 comparison uses the same held-out split. The small smoke run tests mechanics on synthetic images; its scores do not rank photographic model quality.


In [ ]:
import os,json
from pathlib import Path
import numpy as np,pandas as pd,torch
import matplotlib.pyplot as plt
torch.manual_seed(42);torch.set_num_threads(1)
np.random.seed(42)
SMOKE=os.environ.get('COURSE_SMOKE','0')=='1'
results=Path('results');results.mkdir(exist_ok=True)

import time
from fastai.vision.all import *
path=Path('data/synthetic_image_smoke' if SMOKE else 'data/bird_forest')
if not path.exists(): path=results/('image_dataset_smoke' if SMOKE else 'image_dataset_full')
if not path.exists(): raise FileNotFoundError('Run notebook 00 to prepare and persist the image folders.')
settings=json.loads((results/'image_config.json').read_text())
assert settings['synthetic_smoke'] == SMOKE, 'Rerun notebook 00 with the same full/quick setting.'
files=get_image_files(path)
failed=set(verify_images(files));files=L(f for f in files if f not in failed)
splits=RandomSplitter(valid_pct=.2,seed=42)(files)
rows=[]
for name,architecture in [('resnet18',resnet18),('resnet34',resnet34)]:
    set_seed(42)
    block=DataBlock(blocks=(ImageBlock,CategoryBlock),get_items=lambda _:files,
       splitter=IndexSplitter(splits[1]),get_y=parent_label,item_tfms=Resize(64 if SMOKE else 192))
    dls=block.dataloaders(path,bs=8,num_workers=0)
    learn=vision_learner(dls,architecture,pretrained=not SMOKE,metrics=error_rate)
    start=time.perf_counter()
    with learn.no_bar(),learn.no_logging():learn.fine_tune(1 if SMOKE else 3)
    elapsed=time.perf_counter()-start
    probabilities,targets=learn.get_preds(dl=dls.valid)
    targets=targets.reshape(-1)
    rows.append({'model':name,'parameters':sum(p.numel() for p in learn.model.parameters()),
      'seconds':elapsed,'validation_accuracy':float((probabilities.argmax(1)==targets).float().mean()),
      'synthetic_smoke':SMOKE})
table=pd.DataFrame(rows);table.to_csv(results/'image_model_comparison.csv',index=False)
print(table.to_string(index=False))

Save your outputs

Run the following cell after finishing. Choose **File > Save** in Colab to preserve your code and displayed results. Files in `results/` and `models/` persist in Drive when enabled.

In [ ]:
finish_run(COLAB_CONTEXT)